# Words as cross-attention rows, from scratch

Shapes only, random weights. Order: one prompt first, then a batch, then the block.

checkout to commit - d36a4aa19a7786881064709d35f42296a695d72a


In [1]:
import sys

import torch
from einops import rearrange
from torch import nn

from dream_robot.policies.verne.model import P, Verne

sys.path.insert(0, '..')  # nbs/probe.py
from probe import probe, show

torch.manual_seed(0)
net = Verne()
frame, state = torch.randn(1, 4, 16, 32), torch.randn(1, 8)  # canvas latent, joints + gripper
ins = torch.tensor([[0.0, 1, 0, 0, 1, 0]])  # WORDS one-hot: 'green', 'left'

## 1. Today: the whole instruction is one token among the video tokens


In [2]:
patches = rearrange(frame, 'b c (h p) (w q) -> b (h w) (p q c)', p=P, q=P)
parts = net.state_in(state)[:, None], net.instruction_in(ins)[:, None], net.patch_in(patches)
x = torch.cat(parts, 1) + net.video_pos
show(
    [p.shape for p in parts],
    x.shape,
)

[p.shape for p in parts] = [torch.Size([1, 1, 256]), torch.Size([1, 1, 256]), torch.Size([1, 128, 256])]
x.shape = torch.Size([1, 130, 256])


**130 video tokens of width 256: [state, instruction, 128 patches].**

- 'green' + 'left' are summed into **one** token, the same size as one patch.
- After this, every layer is plain self-attention over the 130: a patch only gets the words if it
  happens to attend to token 1.


## 2. How much of a patch's attention lands on the instruction (layer 0)


In [3]:
q, k, v = net.video[0].qkv_of(x)  # each [1, heads, 130 tokens, 32]
w = (q @ k.transpose(-1, -2) / 32**0.5).softmax(-1)  # [1, heads, 130 queries, 130 keys]
show(
    w.shape,
    w[0, :, 2:, 1].mean().item(),
    1 / 130,
)

w.shape = torch.Size([1, 4, 130, 130])
w[0, :, 2:, 1].mean().item() = 0.0077002947218716145
1 / 130 = 0.007692307692307693


**Each patch gives the instruction token 0.0077 of its attention, i.e. 1/130.**

- `w` is [1, 4 heads, 130 queries, 130 keys]: each row is one token's softmax over all 130 keys, summing to 1.
- `w[0, :, 2:, 1]`: every head, queries = the 128 patches, key 1 = the instruction.
- At random init attention is near uniform. To use the words, training must teach every patch to
  single out key 1, competing against 128 patches and the state.


## 3. One prompt: each word becomes its own row


In [4]:
word_in = nn.Embedding(6, 256)  # one learned row per word in WORDS
ids = ins[0].nonzero()[:, 0]  # the words this prompt has
rows = word_in(ids)[None]  # [1 prompt, n words, 256]
show(
    ids,
    rows.shape,
)

ids = tensor([1, 4])
rows.shape = torch.Size([1, 2, 256])


**Word ids [1, 4] = 'green', 'left' → rows [1, 2, 256].**

- `ins[0].nonzero()` lists the positions of the 1s in the one-hot: those are the word ids.
- `word_in(ids)` picks those rows of the table: two separate vectors, not one summed token.
- The row count depends on the prompt: 2 here, 0 for a T1 prompt (no words). Hold that thought for step 5.


In [5]:
@probe
def _():
    show(
        ins,
        ins.shape,
        ins.nonzero(),
        ins.nonzero()[:, 1],
        ids,
        ids.shape,
        rows.shape,
    )

ins = tensor([[0., 1., 0., 0., 1., 0.]])
ins.shape = torch.Size([1, 6])
ins.nonzero() =
tensor([[0, 1],
        [0, 4]])
ins.nonzero()[:, 1] = tensor([1, 4])
ids = tensor([1, 4])
ids.shape = torch.Size([2])
rows.shape = torch.Size([1, 2, 256])


## 4. One prompt: cross-attention, patches ask and words answer


In [6]:
to_q, to_kv = nn.Linear(256, 4 * 32), nn.Linear(256, 2 * 4 * 32)
cq = rearrange(to_q(x[:, 2:]), 'b s (h d) -> b h s d', h=4)  # queries from the 128 patches
ck, cv = rearrange(to_kv(rows), 'b s (n h d) -> n b h s d', n=2, h=4)  # keys, values from words
cw = (cq @ ck.transpose(-1, -2) / 32**0.5).softmax(-1)  # [1, heads, 128 patches, 2 words]
show(
    cq.shape,
    ck.shape,
    cw.shape,
    cw[0, 0, 0],
)

cq.shape = torch.Size([1, 4, 128, 32])
ck.shape = torch.Size([1, 4, 2, 32])
cw.shape = torch.Size([1, 4, 128, 2])
cw[0, 0, 0] = tensor([0.6119, 0.3881], grad_fn=<SelectBackward0>)


**Weights [1, 4 heads, 128 patches, 2 words]; patch 0, head 0: green 0.61, left 0.39.**

- Queries come from the patches, keys and values from the word rows: a separate attention with only 2 keys.
- Each patch's softmax is over the 2 words only, so all of it lands on the words (vs 1/130 in step 2).
- Training no longer has to find the words; it only learns what to take from them.


## 5. A batch: T5 'green left' and T1 (no words)


In [7]:
batch = torch.tensor([[0.0, 1, 0, 0, 1, 0], [0.0, 0, 0, 0, 0, 0]])
rows_t5, rows_t1 = [word_in(prompt.nonzero()[:, 0]) for prompt in batch]
show(
    rows_t5.shape,
    rows_t1.shape,
)

rows_t5.shape = torch.Size([2, 256])
rows_t1.shape = torch.Size([0, 256])


In [8]:
try:
    torch.stack([rows_t5, rows_t1])
except RuntimeError as e:
    print(e)

stack expects each tensor to be equal size, but got [2, 256] at entry 0 and [0, 256] at entry 1


**T5 gives 2 rows, T1 gives 0 rows: `torch.stack` refuses ([2, 256] vs [0, 256]).**

- A batch must be one rectangular tensor [prompts, rows, 256], so every prompt needs the same row count.
- Next: give every prompt the same rows, and deal with the ones it doesn't have.


## 6. Pad: every prompt gets all 6 word rows, absent ones times 0 (T5 alone first)


In [9]:
padded = ins[0][:, None] * word_in.weight  # [6, 1] * [6, 256]: row j scaled by 0 or 1
show(
    ins[0][:, None].shape,
    word_in.weight.shape,
    padded.shape,
    padded.norm(dim=-1),
)

ins[0][:, None].shape = torch.Size([6, 1])
word_in.weight.shape = torch.Size([6, 256])
padded.shape = torch.Size([6, 256])
padded.norm(dim=-1) =
tensor([ 0.0000, 15.0306,  0.0000,  0.0000, 16.4069,  0.0000],
       grad_fn=<LinalgVectorNormBackward0>)


**6 rows for T5: rows 1 (green) and 4 (left) keep their vector (norm ~15-16), the other 4 are zeros.**

- `ins[0][:, None]` is the one-hot as a column [6, 1]; times the table [6, 256] it scales row j by 0 or 1.
- Same as `word_in(ids)` from step 3, plus 4 zero rows where the absent words would be.
- Every prompt now has 6 rows, whatever its words.


In [12]:
@probe
def _():
    show(
        padded[0][:10],
        padded[1][:10],
        word_in.weight[1][:10],
    )

padded[0][:10] =
tensor([0., 0., -0., -0., -0., 0., -0., -0., -0., -0.],
       grad_fn=<SliceBackward0>)
padded[1][:10] =
tensor([-0.3226,  0.1714, -0.5916,  0.0208,  0.0692,  0.2292, -0.1540, -1.4072,
        -0.5169, -0.0391], grad_fn=<SliceBackward0>)
word_in.weight[1][:10] =
tensor([-0.3226,  0.1714, -0.5916,  0.0208,  0.0692,  0.2292, -0.1540, -1.4072,
        -0.5169, -0.0391], grad_fn=<SliceBackward0>)


## 6b. The same pad on the batch


In [13]:
rows_b = batch[:, :, None] * word_in.weight  # [2, 6, 1] * [6, 256] -> [2, 6, 256]
show(
    batch[:, :, None].shape,
    rows_b.shape,
    rows_b.norm(dim=-1),
)

batch[:, :, None].shape = torch.Size([2, 6, 1])
rows_b.shape = torch.Size([2, 6, 256])
rows_b.norm(dim=-1) =
tensor([[ 0.0000, 15.0306,  0.0000,  0.0000, 16.4069,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],
       grad_fn=<LinalgVectorNormBackward0>)


**rows [2 prompts, 6 words, 256]: T5 has rows 1 and 4 real, T1 is all zeros. It stacks.**

- `batch[:, :, None]` [2, 6, 1] broadcasts against the table [6, 256]: each prompt scales its own copy.
- The batch is rectangular now. But a zero row is still a row: the next step asks what attention does with it.


## 7. What the patches read from the padded rows


In [14]:
bk, bv = rearrange(to_kv(rows_b), 'b s (n h d) -> n b h s d', n=2, h=4)  # [2, 4, 6, 32]
bw = (cq @ bk.transpose(-1, -2) / 32**0.5).softmax(-1)  # same patch queries for both prompts
show(
    bk.shape,
    bw.shape,
    bw[:, 0, 0],
)

bk.shape = torch.Size([2, 4, 6, 32])
bw.shape = torch.Size([2, 4, 128, 6])
bw[:, 0, 0] =
tensor([[0.1577, 0.2260, 0.1577, 0.1577, 0.1434, 0.1577],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667]],
       grad_fn=<SelectBackward0>)


**Patch 0, head 0. T5: the 4 zero rows take 0.158 each = 0.63; green 0.23, left 0.14. T1: 1/6 each.**

- A zero row is not invisible: `to_kv(0)` = the layer's bias, a real key, and softmax gives it weight.
- The 4 zero rows are 4 identical keys, so together they soak up most of a T5 patch's attention.
- `cq` [1, 4, 128, 32] broadcasts over the 2 prompts: the same patch queries, different word keys.
- T1 does get a defined answer (no NaN), just a meaningless one. Training could learn to push the
  zero keys down, but it starts diluted.


## 8. Mask: absent words get score -inf, so softmax gives them 0


In [15]:
scores = cq @ bk.transpose(-1, -2) / 32**0.5  # [2, 4, 128, 6]
mask = batch.bool()[:, None, None]  # [2, 1, 1, 6]: True = the prompt has this word
mw = scores.masked_fill(~mask, float('-inf')).softmax(-1)
show(
    mask.shape,
    mw[:, 0, 0],
)

mask.shape = torch.Size([2, 1, 1, 6])
mw[:, 0, 0] =
tensor([[0.0000, 0.6119, 0.0000, 0.0000, 0.3881, 0.0000],
        [   nan,    nan,    nan,    nan,    nan,    nan]],
       grad_fn=<SelectBackward0>)


**T5: green 0.61, left 0.39, absent words 0, exactly step 4's numbers. T1: all NaN.**

- Masking makes the padded batch behave as if each prompt had only its own rows (step 4, unpadded).
- T1 has no words, so every score is -inf: softmax = exp(-inf) / sum of exp(-inf) = 0/0 = NaN.
- One NaN in the batch spreads into the loss and every weight. T1 needs at least one key it may attend to.


## 9. Fix: one learned "no word" row, first, never masked


In [16]:
no_word = nn.Parameter(torch.randn(1, 1, 256))
rows_m = torch.cat([no_word.expand(2, -1, -1), rows_b], 1)  # [2, 7, 256]: [no word, 6 words]
mask_m = torch.cat([torch.ones(2, 1, dtype=torch.bool), batch.bool()], 1)  # [2, 7]
show(
    rows_m.shape,
    mask_m,
)

rows_m.shape = torch.Size([2, 7, 256])
mask_m =
tensor([[ True, False,  True, False, False,  True, False],
        [ True, False, False, False, False, False, False]])


**rows [2, 7, 256] = [no word, 6 words]; mask True at key 0 for both, then at the prompt's words.**

- T5 'green left': keys 0, 2 (green), 5 (left); everything shifts by one because "no word" sits at 0.
- T1: key 0 only. Every prompt now has at least one key it may attend to.


In [17]:
mk, mv = rearrange(to_kv(rows_m), 'b s (n h d) -> n b h s d', n=2, h=4)  # [2, 4, 7, 32]
scores = cq @ mk.transpose(-1, -2) / 32**0.5  # [2, 4, 128, 7]
mw = scores.masked_fill(~mask_m[:, None, None], float('-inf')).softmax(-1)
show(mw[:, 0, 0])

mw[:, 0, 0] =
tensor([[0.2726, 0.0000, 0.4451, 0.0000, 0.0000, 0.2823, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<SelectBackward0>)


**T5: no word 0.27, green 0.45, left 0.28, absent 0. T1: no word 1.0, no NaN.**

- T5 splits its attention over 3 real keys, not diluted by 4 identical zero rows (step 7).
- T1 reads only the "no word" row: a learned "no instruction" signal.
- "No word" also gives a T5 patch a place to park attention when the words don't matter to it.
- Recipe: **pad** so the batch stacks, **mask** so the padding is ignored, **one always-on row** so no prompt is empty.


## 10. One block on the batch: a) self-attention, without the instruction token


In [18]:
import torch.nn.functional as F

xs = torch.cat([x[:, :1], x[:, 2:]], 1).expand(2, -1, -1)  # [state, 128 patches], both prompts
b = net.video[0]
q, k, v = b.qkv_of(xs)
xs = xs + b.out(rearrange(F.scaled_dot_product_attention(q, k, v), 'b h s d -> b s (h d)'))
show(
    xs.shape,
    (xs[0] - xs[1]).abs().max(),
)

xs.shape = torch.Size([2, 129, 256])
(xs[0] - xs[1]).abs().max() = tensor(1.1921e-07, grad_fn=<MaxBackward1>)


**129 tokens [state, 128 patches] for both prompts; the two are identical (diff 1e-7).**

- The same layer as today (`net.video[0]`), one token fewer: the words left the sequence.
- Self-attention can't tell the prompts apart: nothing in it has seen the words yet.


## b) cross-attention to the word rows, then the MLP


In [ ]:
norm3, cross_out = nn.LayerNorm(256), nn.Linear(4 * 32, 256)
cq = rearrange(to_q(norm3(xs)), 'b s (h d) -> b h s d', h=4)  # all 129 tokens ask
a = F.scaled_dot_product_attention(cq, mk, mv, attn_mask=mask_m[:, None, None])
xs = xs + cross_out(rearrange(a, 'b h s d -> b s (h d)'))
show(
    (xs[0] - xs[1]).abs().max(),
)
xs = xs + b.mlp(b.norm2(xs))
show(
    a.shape,
    xs.shape,
    (xs[0] - xs[1]).abs().max(),
)

(xs[0] - xs[1]).abs().max() = tensor(2.2843, grad_fn=<MaxBackward1>)
a.shape = torch.Size([2, 4, 129, 32])
xs.shape = torch.Size([2, 129, 256])
(xs[0] - xs[1]).abs().max() = tensor(2.4905, grad_fn=<MaxBackward1>)


**After cross-attention + MLP: out [2, 129, 256], same as in; the prompts now differ (max diff 1.36).**

- `a` [2, 4, 129, 32]: each token's weighted mix of its prompt's word values (masked, with "no word").
- Cross-attention adds to each token (residual); it doesn't add tokens.
- New pieces per block: `norm3`, `to_q`, `to_kv`, `cross_out`. Order (as in Wan): self → cross → MLP.


## 11. Action side today: the words as one vector added to every action step


In [22]:
chunk, tau = torch.randn(2, 16, 8), torch.rand(2)  # a noisy chunk of 16 actions, its flow time
y = net.action_in(chunk) + net.action_pos + net.time_in(tau[:, None])[:, None]
w_act = net.instruction_act(batch)[:, None]  # [2, 1, 128], broadcast over the 16 steps
show(
    y.shape,
    w_act.shape,
    (y + w_act).shape,
)

y.shape = torch.Size([2, 16, 128])
w_act.shape = torch.Size([2, 1, 128])
(y + w_act).shape = torch.Size([2, 16, 128])


**Action tokens [2, 16, 128]; the words [2, 1, 128], the same vector added to all 16 steps.**

- Step 1 and step 16 of the chunk get identical word information, fixed before any layer runs.


## b) Action side with cross-attention: each step asks its prompt's word rows


In [23]:
act_words = nn.Embedding(7, 128)  # the action side's own table: [no word, 6 words], width 128
rows_a = act_words.weight.expand(2, -1, -1)  # all 7 rows for both prompts; the mask does the rest
a_q, a_kv = nn.Linear(128, 4 * 32), nn.Linear(128, 2 * 4 * 32)
aq = rearrange(a_q(y), 'b s (h d) -> b h s d', h=4)  # queries from the 16 action steps
ak, av = rearrange(a_kv(rows_a), 'b s (n h d) -> n b h s d', n=2, h=4)
aw = (aq @ ak.transpose(-1, -2) / 32**0.5).masked_fill(~mask_m[:, None, None], float('-inf'))
show(
    aw.shape,
    aw.softmax(-1)[:, 0, :2],
)

aw.shape = torch.Size([2, 4, 16, 7])
aw.softmax(-1)[:, 0, :2] =
tensor([[[0.3517, 0.0000, 0.3298, 0.0000, 0.0000, 0.3184, 0.0000],
         [0.2373, 0.0000, 0.4685, 0.0000, 0.0000, 0.2943, 0.0000]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]]],
       grad_fn=<SliceBackward0>)


**Weights [2 prompts, 4 heads, 16 steps, 7 rows]. T5, steps 0-1: no word / green / left = 0.35/0.33/0.32, then 0.24/0.47/0.29. T1: no word 1.0.**

- Each step weighs the words on its own, and again in every layer (its queries change as it denoises).
- The table's rows are all real (no zeroing as in step 6): the mask alone hides absent words, so padding
  needs no multiply once there is a mask.
- Same recipe as the video side (no word + mask), with its own 7-row table at width 128.


## 12. Pack: a) `CrossBlock` = today's `Block` + the cross-attention sub-layer (step 10)


In [28]:
from torch import Tensor

from dream_robot.policies.verne.model import HEAD_DIM, HEADS, Block


class CrossBlock(Block):
    def __init__(self, dim: int):
        super().__init__(dim)
        self.norm3, self.cq = nn.LayerNorm(dim), nn.Linear(dim, HEADS * HEAD_DIM)
        self.ckv, self.cout = nn.Linear(dim, 2 * HEADS * HEAD_DIM), nn.Linear(HEADS * HEAD_DIM, dim)

    def finish(
        self,
        x: Tensor,
        q: Tensor,
        k: Tensor,
        v: Tensor,
        rows: Tensor,
        rmask: Tensor,
        mask: Tensor | None = None,
    ) -> Tensor:
        # rows [B, R, dim] the word rows, rmask [B, R] True = may attend; mask as Block's
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        x = x + self.out(rearrange(a, 'b h s d -> b s (h d)'))
        cq = rearrange(self.cq(self.norm3(x)), 'b s (h d) -> b h s d', h=HEADS)
        ck, cv = rearrange(self.ckv(rows), 'b s (n h d) -> n b h s d', n=2, h=HEADS)
        a = F.scaled_dot_product_attention(cq, ck, cv, attn_mask=rmask[:, None, None])
        x = x + self.cout(rearrange(a, 'b h s d -> b s (h d)'))
        return x + self.mlp(self.norm2(x))

In [29]:
cb = CrossBlock(256)
x2 = torch.cat([x[:, :1], x[:, 2:]], 1).expand(2, -1, -1)  # [2, 129, 256], same tokens
q, k, v = cb.qkv_of(x2)
out = cb.finish(x2, q, k, v, rows_m, mask_m)
n_new = sum(p.numel() for p in cb.parameters()) - sum(p.numel() for p in b.parameters())
show(
    out.shape,
    (out[0] - out[1]).abs().max(),
    n_new,
)

out.shape = torch.Size([2, 129, 256])
(out[0] - out[1]).abs().max() = tensor(1.0698, grad_fn=<MaxBackward1>)
n_new = 132224


**Out [2, 129, 256], prompts differ (max diff 1.07); +132,224 params per video block.**

- The same two steps as 10a + 10b, now inside one layer: `finish` runs self → cross → MLP.
- Extra per video block: `norm3` + `cq` (256→128) + `ckv` (256→256) + `cout` (128→256).


## b) `VerneX`: today's Verne with word rows on both sides (no future branch)


In [30]:
from dream_robot.policies.verne.model import CHUNK, D_ACTION, D_VIDEO, LAYERS, N_PATCH, Z


class VerneX(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_in, self.state_in = nn.Linear(Z * P * P, D_VIDEO), nn.Linear(8, D_VIDEO)
        self.video_pos = nn.Parameter(torch.randn(1, 1 + N_PATCH, D_VIDEO) * 0.02)
        self.video_words = nn.Embedding(7, D_VIDEO)  # row 0 'no word', then WORDS
        self.video = nn.ModuleList([CrossBlock(D_VIDEO) for _ in range(LAYERS)])
        self.action_in, self.action_words = nn.Linear(8, D_ACTION), nn.Embedding(7, D_ACTION)
        self.time_in = nn.Sequential(
            nn.Linear(1, D_ACTION), nn.SiLU(), nn.Linear(D_ACTION, D_ACTION)
        )
        self.action_pos = nn.Parameter(torch.randn(1, CHUNK, D_ACTION) * 0.02)
        self.action = nn.ModuleList([CrossBlock(D_ACTION) for _ in range(LAYERS)])
        self.action_out = nn.Linear(D_ACTION, 8)

    def see(self, frame: Tensor, state: Tensor, ins: Tensor) -> list[tuple[Tensor, Tensor]]:
        # -> each video layer's keys and values; the words enter only through cross-attention
        patches = rearrange(frame, 'b c (h p) (w q) -> b (h w) (p q c)', p=P, q=P)
        x = torch.cat([self.state_in(state)[:, None], self.patch_in(patches)], 1) + self.video_pos
        rows = self.video_words.weight.expand(len(x), -1, -1)  # [B, 7, D_VIDEO]
        rmask = torch.cat([torch.ones_like(ins[:, :1]), ins], 1).bool()  # 'no word' + the prompt's
        kvs: list[tuple[Tensor, Tensor]] = []
        for block in self.video:
            q, k, v = block.qkv_of(x)
            kvs.append((k, v))
            x = block.finish(x, q, k, v, rows, rmask)
        return kvs

    def velocity(
        self, kvs: list[tuple[Tensor, Tensor]], x: Tensor, tau: Tensor, ins: Tensor
    ) -> Tensor:
        y = self.action_in(x) + self.action_pos + self.time_in(tau[:, None])[:, None]
        rows = self.action_words.weight.expand(len(y), -1, -1)  # [B, 7, D_ACTION]
        rmask = torch.cat([torch.ones_like(ins[:, :1]), ins], 1).bool()
        for block, (k_video, v_video) in zip(self.action, kvs):
            q, k, v = block.qkv_of(y)
            k, v = torch.cat([k_video, k], 2), torch.cat([v_video, v], 2)
            y = block.finish(y, q, k, v, rows, rmask)
        return self.action_out(y)

In [31]:
nx = VerneX()
kvs = nx.see(frame.expand(2, -1, -1, -1), state.expand(2, -1), batch)
vel = nx.velocity(kvs, chunk, tau, batch)
today = sum(p.numel() for name, p in net.named_parameters() if not name.startswith('future'))
show(
    len(kvs),
    kvs[0][0].shape,
    vel.shape,
    today,
    sum(p.numel() for p in nx.parameters()),
)

len(kvs) = 4
kvs[0][0].shape = torch.Size([2, 4, 129, 32])
vel.shape = torch.Size([2, 16, 8])
today = 3489928
sum(p.numel() for p in nx.parameters()) = 4283784


**4 video layers of keys/values [2, 4, 129, 32] (was 130 tokens); velocity [2, 16, 8] as today; 4.28M params vs 3.49M (+0.79M, +23%).**

- `see` and `velocity` keep today's interface: the action side still reads each video layer's keys and values.
- Gone: `instruction_in` (the word token), `instruction_act` (the vector added to actions).
  New: two 7-row word tables ([no word, 6 words], one per side) and the cross sub-layer in all 8 blocks.
- `rmask` is steps 8-9 in one line: 'no word' always on, then the prompt's one-hot.
- The +23% is almost all cross-attention projections; the step 08 baseline is 22/50 at 3.49M.
